# Build `model_features.csv`

Merges:

- **`Artifacts/extracted_features.csv`** — 1 Hz rows (`datetime`, `lat`, `lon`, `bc`, weather, and TomTom Flow Segment columns such as `TT_*`).
- **`Code/vehicle_proximity_counts.csv`** — one row per interval (`timestamp` = interval start; counts are aggregates over that window — typically 10 s).

Bin width for aligning environmental data is set by **`BIN_DURATION_SECONDS`** in the next cell (default `10`). Vehicle counts are treated as aggregates over the **preceding** interval (the last *BIN_DURATION_SECONDS* before each `timestamp`). Environmental columns are the **mean** of all 1 Hz samples with `datetime` in **`[timestamp - BIN_DURATION_SECONDS, timestamp)`** (same window as the vehicle counts).

Output rows are restricted to the **overlap** of the two time ranges: vehicle bins whose backward-looking window intersects the shared timeline. Bins with no 1 Hz samples inside that window are dropped (inner merge).

Writes **`Artifacts/model_features.csv`** (all columns from both inputs; `datetime` is set equal to `timestamp` for each row).

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)

def _repo_root() -> Path:
    """Resolve repo root whether the notebook cwd is `Code/` or the repository root."""
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name == "Code" else cwd

ROOT = _repo_root()
ENV_PATH = ROOT / "Artifacts" / "extracted_features.csv"
VEHICLE_PATH = ROOT / "Artifacts" / "vehicle_proximity_counts.csv"
OUT_PATH = ROOT / "Artifacts" / "model_features.csv"

BIN_DURATION_SECONDS = 30

ENV_AGG_MEAN = [
    "lat",
    "lon",
    "bc",
    "Temperature",
    "Humidity",
    "Wind_speed",
    "TT_currentSpeed",
    "TT_freeFlowSpeed",
    "TT_currentTravelTime",
    "TT_freeFlowTravelTime",
    "TT_confidence",
]
ENV_AGG_FIRST = ["TT_frc"]
ENV_AGG_MAX = ["TT_roadClosure", "bc_categorical"]

ENV_OUTPUT_COLS = [
    "lat",
    "lon",
    "bc",
    "bc_categorical",
    "Temperature",
    "Humidity",
    "Wind_speed",
    "TT_frc",
    "TT_currentSpeed",
    "TT_freeFlowSpeed",
    "TT_currentTravelTime",
    "TT_freeFlowTravelTime",
    "TT_confidence",
    "TT_roadClosure",
]

In [ ]:
env = pd.read_csv(ENV_PATH, parse_dates=["datetime"])
veh = pd.read_csv(VEHICLE_PATH, parse_dates=["timestamp"])

if env["datetime"].dt.tz is None:
    env["datetime"] = env["datetime"].dt.tz_localize("UTC")
if veh["timestamp"].dt.tz is None:
    veh["timestamp"] = veh["timestamp"].dt.tz_localize("UTC")

env = env.sort_values("datetime").reset_index(drop=True)
veh = veh.sort_values("timestamp").reset_index(drop=True)

if "bc" in env.columns:
    env["bc"] = env["bc"].where(env["bc"] >= 0)
    env["bc"] = env["bc"].clip(upper=1_000_000)
    bc_valid = env["bc"].dropna()
    if len(bc_valid) > 0:
        q1 = float(bc_valid.quantile(0.25))
        q3 = float(bc_valid.quantile(0.75))

        def _bc_bucket(v: float) -> object:
            if pd.isna(v):
                return pd.NA
            if v <= q1:
                return "low"
            if v <= q3:
                return "normal"
            return "high"

        env["bc_categorical"] = env["bc"].map(_bc_bucket)
        env["bc_categorical"] = pd.Categorical(
            env["bc_categorical"],
            categories=["low", "normal", "high"],
            ordered=True,
        )
    else:
        env["bc_categorical"] = pd.Categorical.from_codes(
            [-1] * len(env),
            categories=["low", "normal", "high"],
            ordered=True,
        )
        print("Warning: no valid `bc` after preprocessing; `bc_categorical` is all missing.")
else:
    raise KeyError("extracted_features.csv must contain a `bc` column.")

print("Env rows:", len(env), "|", env["datetime"].iloc[0], "→", env["datetime"].iloc[-1])
print("Vehicle rows:", len(veh), "|", veh["timestamp"].iloc[0], "→", veh["timestamp"].iloc[-1])

In [ ]:
bin_origin = veh["timestamp"].iloc[0]
bin_seconds = pd.Timedelta(seconds=BIN_DURATION_SECONDS)
print("Bin duration:", BIN_DURATION_SECONDS, "s")

env["_bin"] = ((env["datetime"] - bin_origin) // bin_seconds).astype("int64")

agg_map = {}
for c in ENV_AGG_MEAN:
    if c in env.columns:
        agg_map[c] = "mean"
for c in ENV_AGG_FIRST:
    if c in env.columns:
        agg_map[c] = "first"
for c in ENV_AGG_MAX:
    if c in env.columns:
        agg_map[c] = "max"
if not agg_map:
    raise ValueError("No environment columns to aggregate; check extracted_features.csv schema.")

env_avg = env.groupby("_bin", sort=True).agg(agg_map).reset_index()

veh["_bin"] = ((veh["timestamp"] - bin_origin) // bin_seconds).astype("int64")

merged = veh.merge(env_avg, on="_bin", how="inner")

env_start = env["datetime"].min()
env_end = env["datetime"].max()
veh_start = veh["timestamp"].min()
veh_end = veh["timestamp"].max() + bin_seconds

overlap_start = max(env_start, veh_start)
overlap_end = min(env_end, veh_end)

ts = merged["timestamp"]
in_overlap = (ts + bin_seconds > overlap_start) & (ts < overlap_end)
merged = merged.loc[in_overlap].copy()

merged["datetime"] = merged["timestamp"]
merged = merged.drop(columns=["_bin"])

vehicle_cols = [c for c in veh.columns if c != "_bin"]
env_out = [c for c in ENV_OUTPUT_COLS if c in merged.columns]
missing_env = [c for c in ENV_OUTPUT_COLS if c not in merged.columns]
if missing_env:
    print("Warning: expected env columns missing from merge (regenerate extracted_features?):", missing_env)
out_cols = vehicle_cols + ["datetime"] + env_out
out = merged[out_cols]

print("Overlap:", overlap_start, "→", overlap_end)
print("Output rows:", len(out))
out.head()

In [ ]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(OUT_PATH, index=False)
print("Wrote", OUT_PATH)